<a href="https://colab.research.google.com/github/PattiAlves/Modelagem_IA/blob/main/Notebooks/3.%20RECONSTRU%C3%87%C3%83O%20S%C3%89RIE%20DE%20DADOS%20N%C3%8DVEL%20LAGOA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **3. RECONSTRUÇÃO DA SÉRIE DE DADOS DO NÍVEL DA LAGOA**

### **3.1 Planilha unificada de chuva e maré para o período de set/2017 a set/2026**

In [4]:
import os
import numpy as np
import pandas as pd

try:
  from google.colab import drive

  if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
except Exception:
  pass

# 1. Caminhos e janela temporal
pasta_base = '/content/drive/MyDrive/PUC/EQM2118'
pasta_consolidados = os.path.join(pasta_base, 'CONSOLIDADOS')
pasta_chuva = os.path.join(pasta_base, 'CHUVA_2010_2026')
os.makedirs(pasta_consolidados, exist_ok=True)

caminho_mare = os.path.join(
    pasta_consolidados, 'mare_consolidada_10min_utc3.csv'
)
caminho_chuva = os.path.join(
    pasta_chuva, 'banco_precipitacoes_completo_2010_2026_NaN_preenchido.csv'
)
caminho_saida = os.path.join(
    pasta_consolidados, 'chuva_mare_10min_092017_092026.csv'
)

DATA_INICIO = pd.Timestamp('2017-09-16 18:40:00')
DATA_FIM = pd.Timestamp('2026-09-19 23:00:00')

# Se quiser forçar manualmente o nome exato de alguma coluna após ver a listagem, preencha aqui (ou deixe None)
COLUNA_DATA_CHUVA_FORCADA = None
COLUNA_CHUVA_FORCADA = None
COLUNA_DATA_MARE_FORCADA = None
COLUNA_MARE_FORCADA = None

CANDIDATOS_DATA = [
    'Date_time',
    'datetime',
    'data_hora',
    'Data_Hora',
    'timestamp',
    'date',
    'data',
    'time',
]
CANDIDATOS_CHUVA = [
    'precip_10min',
    'chuva_10min_mm',
    'precipitacao_10min_mm',
    'chuva_mm',
    'precipitacao_mm',
    'chuva',
    'precipitacao',
    'valor',
]
CANDIDATOS_MARE = [
    'mare_10min_m',
    'mare_m',
    'nivel_mare_m',
    'altura_mare_m',
    'mare',
    'nivel',
    'valor',
    'nivel_mar_total_m',
]


def ler_csv_inspecionando_colunas(caminho, rotulo):
  if not os.path.exists(caminho):
    raise FileNotFoundError(f'Arquivo não encontrado: {caminho}')
  df = pd.read_csv(caminho)
  if df.shape[1] == 1:
    df = pd.read_csv(caminho, sep=';')
  print(f'[{rotulo}] Arquivo: {os.path.basename(caminho)}')
  print(f'   • Colunas lidas ({len(df.columns)}): {list(df.columns)}')
  return df


def escolher_coluna(colunas, forçada, candidatos, rotulo):
  if forçada is not None:
    if forçada not in colunas:
      raise ValueError(
          f'Coluna forçada "{forçada}" não existe em {rotulo}. Disponíveis:'
          f' {list(colunas)}'
      )
    return forçada
  for cand in candidatos:
    if cand in colunas:
      return cand
  mapa_lower = {c.lower(): c for c in colunas}
  for cand in candidatos:
    if cand.lower() in mapa_lower:
      return mapa_lower[cand.lower()]
  raise ValueError(
      f'Não foi possível identificar automaticamente a coluna de {rotulo}.'
      f' Colunas disponíveis: {list(colunas)}'
  )


def extrair_serie_numerica(df, col_dt, col_val, nome_saida, rotulo):
  print(f'   • Usando data/hora: "{col_dt}" | valor: "{col_val}" ({rotulo})')
  dt_parsed = pd.to_datetime(df[col_dt], errors='coerce')
  if hasattr(dt_parsed.dt, 'tz') and dt_parsed.dt.tz is not None:
    dt_parsed = dt_parsed.dt.tz_localize(None)

  s = pd.Series(
      pd.to_numeric(df[col_val], errors='coerce').values, index=dt_parsed
  )
  s = s[s.index.notna()].sort_index()
  s = s[~s.index.duplicated(keep='last')]
  s.name = nome_saida
  return s


# 2. Ler colunas e extrair séries de chuva e maré
df_chuva_raw = ler_csv_inspecionando_colunas(caminho_chuva, 'CHUVA')
col_dt_chuva = escolher_coluna(
    df_chuva_raw.columns,
    COLUNA_DATA_CHUVA_FORCADA,
    CANDIDATOS_DATA,
    'data/hora da chuva',
)
col_val_chuva = escolher_coluna(
    df_chuva_raw.columns,
    COLUNA_CHUVA_FORCADA,
    CANDIDATOS_CHUVA,
    'chuva',
)
serie_chuva = extrair_serie_numerica(
    df_chuva_raw, col_dt_chuva, col_val_chuva, 'chuva_10min_mm', 'CHUVA'
)

df_mare_raw = ler_csv_inspecionando_colunas(caminho_mare, 'MARÉ')
col_dt_mare = escolher_coluna(
    df_mare_raw.columns,
    COLUNA_DATA_MARE_FORCADA,
    CANDIDATOS_DATA,
    'data/hora da maré',
)
col_val_mare = escolher_coluna(
    df_mare_raw.columns,
    COLUNA_MARE_FORCADA,
    CANDIDATOS_MARE,
    'maré',
)
serie_mare = extrair_serie_numerica(
    df_mare_raw, col_dt_mare, col_val_mare, 'mare_10min_m', 'MARÉ'
)

# 3. Alinhar em grade contínua de 10 min (incluindo até 24h antes de DATA_INICIO para acumulados e lags)
# Maior janela necessária = 24h = 144 passos de 10 min
inicio_calculo = min(
    DATA_INICIO - pd.Timedelta(hours=24),
    serie_chuva.index.min(),
    serie_mare.index.min(),
)
grade_calculo = pd.date_range(start=inicio_calculo, end=DATA_FIM, freq='10min')

chuva_grade = serie_chuva.reindex(grade_calculo)
mare_grade = serie_mare.reindex(grade_calculo)

# 4. Construir exatamente as mesmas variáveis preditoras do treino/teste (sem interpolar)
feat = pd.DataFrame(index=grade_calculo)
feat['chuva_10min_mm'] = chuva_grade
feat['mare_10min_m'] = mare_grade

janelas_chuva = {
    'chuva_acum_30min_mm': 3,
    'chuva_acum_1h_mm': 6,
    'chuva_acum_3h_mm': 18,
    'chuva_acum_6h_mm': 36,
    'chuva_acum_12h_mm': 72,
    'chuva_acum_24h_mm': 144,
}
for nome_col, n_passos in janelas_chuva.items():
  feat[nome_col] = chuva_grade.rolling(n_passos, min_periods=n_passos).sum()

lags_passos = [1, 3, 6, 12, 18, 36]
for p in lags_passos:
  minutos = p * 10
  feat[f'chuva_lag_{minutos}min'] = chuva_grade.shift(p)
  feat[f'mare_lag_{minutos}min'] = mare_grade.shift(p)

feat['mare_diff_1h'] = mare_grade - mare_grade.shift(6)
feat['mare_diff_3h'] = mare_grade - mare_grade.shift(18)

# 5. Recortar o período exato solicitado e exportar mantendo "Date_time" como primeira coluna
grade_alvo = pd.date_range(start=DATA_INICIO, end=DATA_FIM, freq='10min')
df_final = feat.reindex(grade_alvo).copy()
df_final.insert(0, 'Date_time', grade_alvo.strftime('%Y-%m-%d %H:%M:%S'))

df_final.to_csv(caminho_saida, index=False)

n_total = len(df_final)
n_completas = int(df_final.drop(columns=['Date_time']).notna().all(axis=1).sum())

print('\n=== CONSOLIDAÇÃO COM FEATURES CONCLUÍDA (SEM INTERPOLAÇÃO) ===')
print(f'  • Período: {DATA_INICIO} até {DATA_FIM} (freq = 10min)')
print(f'  • Total de linhas na grade: {n_total}')
print(f'  • Total de colunas exportadas: {len(df_final.columns)}')
print(
    f'  • Linhas com todas as features válidas: {n_completas}'
    f' ({n_completas / n_total * 100:.2f}%)'
)
print(f'  • Arquivo salvo em: {caminho_saida}')

[CHUVA] Arquivo: banco_precipitacoes_completo_2010_2026_NaN_preenchido.csv
   • Colunas lidas (5): ['datetime', 'precip_10min', 'precip_1h', 'flag_preenchido_RC_10min', 'flag_preenchido_RC_1h']
   • Usando data/hora: "datetime" | valor: "precip_10min" (CHUVA)
[MARÉ] Arquivo: mare_consolidada_10min_utc3.csv
   • Colunas lidas (5): ['time', 'surf_el', 'contem_mare', 'mare_astro_nm_m', 'nivel_mar_total_m']
   • Usando data/hora: "time" | valor: "nivel_mar_total_m" (MARÉ)

=== CONSOLIDAÇÃO COM FEATURES CONCLUÍDA (SEM INTERPOLAÇÃO) ===
  • Período: 2017-09-16 18:40:00 até 2026-09-19 23:00:00 (freq = 10min)
  • Total de linhas na grade: 473787
  • Total de colunas exportadas: 23
  • Linhas com todas as features válidas: 473499 (99.94%)
  • Arquivo salvo em: /content/drive/MyDrive/PUC/EQM2118/CONSOLIDADOS/chuva_mare_10min_092017_092026.csv


### **3.2 Reconstrução da série de nível da lagoa de set/2017 a set/2026**

In [5]:
import glob
import os
import joblib
import numpy as np
import pandas as pd

try:
  from google.colab import drive

  if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
except Exception:
  pass

# 1. Caminhos e janela temporal da reconstrução
pasta_base = '/content/drive/MyDrive/PUC/EQM2118'
pasta_consolidados = os.path.join(pasta_base, 'CONSOLIDADOS')
pasta_modelo_final = os.path.join(pasta_consolidados, 'MODELO_FINAL_RNA')
pasta_saida = os.path.join(pasta_consolidados, 'SERIE_LAGOA_RECONST')
os.makedirs(pasta_saida, exist_ok=True)

DATA_INICIO_RECONST = pd.Timestamp('2017-09-16 18:40:00')
DATA_FIM_RECONST = pd.Timestamp('2026-09-19 23:00:00')

caminho_chuva_mare = os.path.join(
    pasta_consolidados, 'chuva_mare_10min_092017_092026.csv'
)
caminho_base_historica = os.path.join(
    pasta_consolidados, 'tabela_unificada_apenas_10min_ate_20170916_1840.csv'
)

# 2. Carregar pacote do treino final (.joblib mais recente em MODELO_FINAL_RNA)
arquivos_joblib = sorted(
    glob.glob(os.path.join(pasta_modelo_final, '*_FINAL.joblib'))
)
if not arquivos_joblib:
  raise FileNotFoundError(
      f'Nenhum arquivo *_FINAL.joblib encontrado em {pasta_modelo_final}.'
      ' Rode o treino final primeiro.'
  )

caminho_modelo_final = arquivos_joblib[-1]
pacote = joblib.load(caminho_modelo_final)
mlp_final = pacote['modelo']
scaler_final = pacote['scaler']
feature_cols = pacote['features']

print(f'Modelo final carregado: {caminho_modelo_final}')
print(f'Número de features esperadas: {len(feature_cols)}')

# 3. Carregar arquivo consolidado chuva_mare_10min_092017_092026.csv
if not os.path.exists(caminho_chuva_mare):
  raise FileNotFoundError(
      f'Arquivo consolidado não encontrado em: {caminho_chuva_mare}'
  )

df_cm = pd.read_csv(caminho_chuva_mare)
col_dt = (
    'Date_time'
    if 'Date_time' in df_cm.columns
    else ('datetime' if 'datetime' in df_cm.columns else df_cm.columns[0])
)

df_cm['datetime'] = pd.to_datetime(df_cm[col_dt], errors='coerce')
df_cm = (
    df_cm.dropna(subset=['datetime'])
    .sort_values('datetime')
    .drop_duplicates(subset=['datetime'], keep='last')
    .set_index('datetime')
)

# 4. Montar grade contínua de 10 min e garantir todas as features (sem interpolar)
grade_alvo = pd.date_range(
    start=DATA_INICIO_RECONST, end=DATA_FIM_RECONST, freq='10min'
)

if set(feature_cols).issubset(set(df_cm.columns)):
  # Caso o arquivo já contenha todas as features calculadas
  feat_reconst = df_cm.reindex(grade_alvo)[feature_cols].apply(
      pd.to_numeric, errors='coerce'
  )
else:
  # Caso o arquivo contenha apenas chuva_10min_mm e mare_10min_m (ou nomes equivalentes)
  col_chuva = (
      'chuva_10min_mm'
      if 'chuva_10min_mm' in df_cm.columns
      else ('precip_10min' if 'precip_10min' in df_cm.columns else None)
  )
  col_mare = (
      'mare_10min_m'
      if 'mare_10min_m' in df_cm.columns
      else (
          'nivel_mar_total_m'
          if 'nivel_mar_total_m' in df_cm.columns
          else None
      )
  )

  if col_chuva is None or col_mare is None:
    raise ValueError(
        'Não foi possível localizar colunas de chuva e maré no arquivo'
        f' consolidado. Colunas disponíveis: {list(df_cm.columns)}'
    )

  s_chuva = pd.to_numeric(df_cm[col_chuva], errors='coerce')
  s_mare = pd.to_numeric(df_cm[col_mare], errors='coerce')

  # Inclui histórico anterior (até 24h antes de DATA_INICIO_RECONST) para aquecer acumulados e lags
  if os.path.exists(caminho_base_historica):
    df_hist = pd.read_csv(caminho_base_historica)
    col_dt_h = 'datetime' if 'datetime' in df_hist.columns else df_hist.columns[0]
    df_hist['datetime'] = pd.to_datetime(df_hist[col_dt_h], errors='coerce')
    df_hist = (
        df_hist.dropna(subset=['datetime'])
        .sort_values('datetime')
        .drop_duplicates(subset=['datetime'], keep='last')
        .set_index('datetime')
    )
    if 'chuva_10min_mm' in df_hist.columns:
      s_chuva = pd.concat(
          [pd.to_numeric(df_hist['chuva_10min_mm'], errors='coerce'), s_chuva]
      )
    if 'mare_10min_m' in df_hist.columns:
      s_mare = pd.concat(
          [pd.to_numeric(df_hist['mare_10min_m'], errors='coerce'), s_mare]
      )

    s_chuva = s_chuva[~s_chuva.index.duplicated(keep='last')].sort_index()
    s_mare = s_mare[~s_mare.index.duplicated(keep='last')].sort_index()

  inicio_calc = min(
      DATA_INICIO_RECONST - pd.Timedelta(hours=24),
      s_chuva.index.min(),
      s_mare.index.min(),
  )
  grade_calc = pd.date_range(
      start=inicio_calc, end=DATA_FIM_RECONST, freq='10min'
  )

  chuva_g = s_chuva.reindex(grade_calc)
  mare_g = s_mare.reindex(grade_calc)

  feat_calc = pd.DataFrame(index=grade_calc)
  feat_calc['chuva_10min_mm'] = chuva_g
  feat_calc['mare_10min_m'] = mare_g

  janelas_chuva = {
      'chuva_acum_30min_mm': 3,
      'chuva_acum_1h_mm': 6,
      'chuva_acum_3h_mm': 18,
      'chuva_acum_6h_mm': 36,
      'chuva_acum_12h_mm': 72,
      'chuva_acum_24h_mm': 144,
  }
  for nome_col, n_passos in janelas_chuva.items():
    feat_calc[nome_col] = chuva_g.rolling(n_passos, min_periods=n_passos).sum()

  lags_passos = [1, 3, 6, 12, 18, 36]
  for p in lags_passos:
    minutos = p * 10
    feat_calc[f'chuva_lag_{minutos}min'] = chuva_g.shift(p)
    feat_calc[f'mare_lag_{minutos}min'] = mare_g.shift(p)

  feat_calc['mare_diff_1h'] = mare_g - mare_g.shift(6)
  feat_calc['mare_diff_3h'] = mare_g - mare_g.shift(18)

  feat_reconst = feat_calc.reindex(grade_alvo)[feature_cols]

# 5. Reconstruir apenas onde todas as features são válidas (sem interpolar)
na_lagoa_reconst = pd.Series(np.nan, index=grade_alvo, dtype=float)
mascara_valida = feat_reconst.notna().all(axis=1)

if mascara_valida.any():
  X_reconst_scaled = scaler_final.transform(
      feat_reconst.loc[mascara_valida].values
  )
  na_lagoa_reconst.loc[mascara_valida] = mlp_final.predict(X_reconst_scaled)

# 6. Exportar DataFrame final com colunas exatas "Date_time" e "NA_lagoa"
df_saida = pd.DataFrame({
    'Date_time': grade_alvo.strftime('%Y-%m-%d %H:%M:%S'),
    'NA_lagoa': na_lagoa_reconst.values,
})

nome_saida = (
    'serie_lagoa_reconstituida_10min_20170916_1840_a_20260919_2300.csv'
)
caminho_saida = os.path.join(pasta_saida, nome_saida)
df_saida.to_csv(caminho_saida, index=False)

n_total = len(df_saida)
n_val = int(mascara_valida.sum())
n_nan = int((~mascara_valida).sum())

print('\n=== RECONSTRUÇÃO CONCLUÍDA (SEM INTERPOLAÇÃO) ===')
print(f'  • Período: {DATA_INICIO_RECONST} até {DATA_FIM_RECONST} (freq = 10min)')
print(f'  • Total de registros na grade: {n_total}')
print(
    f'  • Registros reconstruídos: {n_val} ({n_val / n_total * 100:.2f}%) |'
    f' NaN: {n_nan}'
)
if n_val > 0:
  print(
      '  • NA_lagoa reconstruído (m): min ='
      f' {na_lagoa_reconst.min():.4f} | média ='
      f' {na_lagoa_reconst.mean():.4f} | máx ='
      f' {na_lagoa_reconst.max():.4f}'
  )
print(f'  • Arquivo salvo em: {caminho_saida}')

Modelo final carregado: /content/drive/MyDrive/PUC/EQM2118/CONSOLIDADOS/MODELO_FINAL_RNA/MLPRegressor_10min_acum_lags_3camadas_iter350_FINAL.joblib
Número de features esperadas: 22

=== RECONSTRUÇÃO CONCLUÍDA (SEM INTERPOLAÇÃO) ===
  • Período: 2017-09-16 18:40:00 até 2026-09-19 23:00:00 (freq = 10min)
  • Total de registros na grade: 473787
  • Registros reconstruídos: 473499 (99.94%) | NaN: 288
  • NA_lagoa reconstruído (m): min = -0.2250 | média = 0.3111 | máx = 1.9112
  • Arquivo salvo em: /content/drive/MyDrive/PUC/EQM2118/CONSOLIDADOS/SERIE_LAGOA_RECONST/serie_lagoa_reconstituida_10min_20170916_1840_a_20260919_2300.csv


### **3.3 Análise dos dados da série reconstruída**

In [6]:
import glob
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
  from google.colab import drive

  if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
except Exception:
  pass

# 1. Caminhos
pasta_base = '/content/drive/MyDrive/PUC/EQM2118'
pasta_consolidados = os.path.join(pasta_base, 'CONSOLIDADOS')
pasta_reconst = os.path.join(pasta_consolidados, 'SERIE_LAGOA_RECONST')
pasta_eda = os.path.join(pasta_reconst, 'ANALISE_EXPLORATORIA')
os.makedirs(pasta_eda, exist_ok=True)

# Localiza o CSV reconstruído mais recente na pasta SERIE_LAGOA_RECONST
arquivos_reconst = sorted(
    glob.glob(os.path.join(pasta_reconst, 'serie_lagoa_reconstituida_*.csv'))
)
if not arquivos_reconst:
  raise FileNotFoundError(
      f'Nenhum arquivo reconstruído encontrado em {pasta_reconst}'
  )
caminho_reconst = arquivos_reconst[-1]

caminho_hist = os.path.join(
    pasta_consolidados, 'tabela_unificada_apenas_10min_ate_20170916_1840.csv'
)

# 2. Carregar série reconstruída e alinhar na grade de 10 min sem interpolar
df_rec = pd.read_csv(caminho_reconst)
col_dt_rec = 'Date_time' if 'Date_time' in df_rec.columns else df_rec.columns[0]
col_na_rec = 'NA_lagoa' if 'NA_lagoa' in df_rec.columns else df_rec.columns[1]

df_rec['dt'] = pd.to_datetime(df_rec[col_dt_rec], errors='coerce')
df_rec = (
    df_rec.dropna(subset=['dt'])
    .sort_values('dt')
    .drop_duplicates(subset=['dt'], keep='last')
    .set_index('dt')
)

grade_rec = pd.date_range(
    start=df_rec.index.min(), end=df_rec.index.max(), freq='10min'
)
s_rec = pd.to_numeric(df_rec[col_na_rec], errors='coerce').reindex(grade_rec)
s_rec.name = 'NA_lagoa_reconst'

# Carregar base histórica (opcional, para comparação)
s_hist = None
if os.path.exists(caminho_hist):
  df_h = pd.read_csv(caminho_hist)
  col_dt_h = 'datetime' if 'datetime' in df_h.columns else df_h.columns[0]
  df_h['dt'] = pd.to_datetime(df_h[col_dt_h], errors='coerce')
  df_h = (
      df_h.dropna(subset=['dt'])
      .sort_values('dt')
      .drop_duplicates(subset=['dt'], keep='last')
      .set_index('dt')
  )
  if 'NA_lagoa_10min_m' in df_h.columns:
    s_hist = pd.to_numeric(df_h['NA_lagoa_10min_m'], errors='coerce')
    s_hist.name = 'NA_lagoa_hist'


def resumo_serie(serie, nome):
  s_val = serie.dropna()
  return {
      'serie': nome,
      'total_grade_10min': int(len(serie)),
      'validos': int(len(s_val)),
      'n_nan': int(serie.isna().sum()),
      'pct_nan': round(float(serie.isna().mean() * 100), 4),
      'media_m': round(float(s_val.mean()), 4),
      'dp_m': round(float(s_val.std(ddof=1)), 4),
      'min_m': round(float(s_val.min()), 4),
      'p01_m': round(float(np.percentile(s_val, 1)), 4),
      'p05_m': round(float(np.percentile(s_val, 5)), 4),
      'mediana_p50_m': round(float(np.percentile(s_val, 50)), 4),
      'p95_m': round(float(np.percentile(s_val, 95)), 4),
      'p99_m': round(float(np.percentile(s_val, 99)), 4),
      'max_m': round(float(s_val.max()), 4),
  }


# 3. Resumo geral (reconstruída vs. histórica)
linhas_resumo = [resumo_serie(s_rec, 'Reconstruida_2017_2026')]
if s_hist is not None:
  linhas_resumo.append(resumo_serie(s_hist, 'Historica_ate_2017'))

df_resumo = pd.DataFrame(linhas_resumo)
df_resumo.to_csv(
    os.path.join(pasta_eda, 'resumo_geral_reconstruida.csv'), index=False
)

print('=== RESUMO GERAL ===')
print(df_resumo.to_string(index=False))

# 4. Estatísticas anuais
df_tmp = pd.DataFrame({'NA_lagoa': s_rec, 'ano': s_rec.index.year})
resumo_anual = (
    df_tmp.groupby('ano')['NA_lagoa']
    .agg(
        total_grade='size',
        validos='count',
        n_nan=lambda x: int(x.isna().sum()),
        media_m='mean',
        dp_m='std',
        min_m='min',
        p50_m='median',
        p95_m=lambda x: np.nanpercentile(x, 95) if x.notna().any() else np.nan,
        max_m='max',
    )
    .reset_index()
)
resumo_anual.to_csv(
    os.path.join(pasta_eda, 'estatisticas_anuais_reconstruida.csv'), index=False
)

print('\n=== ESTATÍSTICAS ANUAIS (RECONSTRUÍDA) ===')
print(resumo_anual.round(4).to_string(index=False))

# 5. Sazonalidade mensal (média, mediana, p95 e máximo por mês do ano)
df_tmp['mes'] = s_rec.index.month
resumo_mensal = (
    df_tmp.groupby('mes')['NA_lagoa']
    .agg(
        validos='count',
        n_nan=lambda x: int(x.isna().sum()),
        media_m='mean',
        dp_m='std',
        p50_m='median',
        p95_m=lambda x: np.nanpercentile(x, 95) if x.notna().any() else np.nan,
        max_m='max',
    )
    .reset_index()
)
resumo_mensal.to_csv(
    os.path.join(pasta_eda, 'sazonalidade_mensal_reconstruida.csv'), index=False
)

# 6. Limiares e top extremos
limiares = [0.80, 1.00, 1.20, 1.50]
print('\n=== FREQUÊNCIA ACIMA DE LIMIARES (RECONSTRUÍDA) ===')
s_val = s_rec.dropna()
for lim in limiares:
  qtd = int((s_val > lim).sum())
  pct = (qtd / len(s_val) * 100) if len(s_val) > 0 else 0.0
  print(f'  • NA_lagoa > {lim:.2f} m: {qtd} registros ({pct:.3f}%)')

top_max = s_rec.sort_values(ascending=False).head(15).reset_index()
top_max.columns = ['Date_time', 'NA_lagoa']
top_min = s_rec.sort_values(ascending=True).head(15).reset_index()
top_min.columns = ['Date_time', 'NA_lagoa']

top_max.to_csv(os.path.join(pasta_eda, 'top15_maiores_niveis.csv'), index=False)
top_min.to_csv(os.path.join(pasta_eda, 'top15_menores_niveis.csv'), index=False)

# Datas exatas dos registros com NaN
datas_nan = s_rec.index[s_rec.isna()].to_series().reset_index(drop=True)
datas_nan.to_frame(name='Date_time').to_csv(
    os.path.join(pasta_eda, 'registros_nan_reconstruida.csv'), index=False
)

# 7. Gráficos exploratórios
plt.rcParams['figure.dpi'] = 140

# 7a. Série temporal reconstruída
fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(
    s_rec.index,
    s_rec.values,
    linewidth=0.45,
    color='#1f77b4',
    label='NA_lagoa reconstruído (10 min)',
)
if s_rec.isna().any():
  ax.scatter(
      s_rec.index[s_rec.isna()],
      np.repeat(s_rec.min(), s_rec.isna().sum()),
      color='crimson',
      s=8,
      alpha=0.7,
      label='Falhas / NaN (sem interpolação)',
  )
ax.axhline(
    s_val.mean(),
    color='black',
    linestyle='--',
    linewidth=1.0,
    label=f'Média ({s_val.mean():.3f} m)',
)
ax.set_title(
    'Série Reconstruída do Nível da Lagoa (2017–2026) — Grade de 10 min'
)
ax.set_ylabel('Nível (m)')
ax.set_xlabel('Data / Hora')
ax.legend(loc='upper right', frameon=True)
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(pasta_eda, 'serie_temporal_reconstruida.png'))
plt.close()

# 7b. Média anual e média mensal
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.3))

ax1.bar(
    resumo_anual['ano'].astype(str),
    resumo_anual['media_m'],
    color='#2b8cbe',
    edgecolor='black',
    linewidth=0.5,
)
ax1.set_title('Média Anual do NA_lagoa Reconstruído')
ax1.set_ylabel('Nível médio (m)')
ax1.tick_params(axis='x', rotation=45)
ax1.grid(True, axis='y', alpha=0.25)

ax2.plot(
    resumo_mensal['mes'],
    resumo_mensal['media_m'],
    marker='o',
    color='#08519c',
    label='Média mensal',
)
ax2.plot(
    resumo_mensal['mes'],
    resumo_mensal['p95_m'],
    marker='s',
    linestyle='--',
    color='#e6550d',
    label='Percentil 95',
)
ax2.set_title('Sazonalidade Mensal (Média e P95)')
ax2.set_xlabel('Mês do ano')
ax2.set_ylabel('Nível (m)')
ax2.set_xticks(range(1, 13))
ax2.legend()
ax2.grid(True, alpha=0.25)

plt.tight_layout()
plt.savefig(os.path.join(pasta_eda, 'medias_anuais_e_sazonalidade_mensal.png'))
plt.close()

print(f'\nAnálise exploratória concluída. Arquivos salvos em: {pasta_eda}')

=== RESUMO GERAL ===
                 serie  total_grade_10min  validos  n_nan  pct_nan  media_m   dp_m  min_m   p01_m  p05_m  mediana_p50_m  p95_m  p99_m  max_m
Reconstruida_2017_2026             473787   473499    288   0.0608   0.3111 0.1206 -0.225  0.0603 0.1532         0.2939 0.5314 0.6835 1.9112
    Historica_ate_2017             239199   205879  33320  13.9298   0.3028 0.1631 -0.135 -0.0240 0.0770         0.2880 0.5770 0.7670 2.5560

=== ESTATÍSTICAS ANUAIS (RECONSTRUÍDA) ===
 ano  total_grade  validos  n_nan  media_m   dp_m   min_m  p50_m  p95_m  max_m
2017        15296    15296      0   0.2929 0.0944 -0.0209 0.2840 0.4499 1.2013
2018        52560    52560      0   0.3194 0.1132 -0.0275 0.3006 0.5283 1.6874
2019        52560    52560      0   0.3167 0.1214 -0.0299 0.2983 0.5169 1.9112
2020        52704    52704      0   0.3108 0.1102 -0.0166 0.2918 0.5257 1.4661
2021        52560    52560      0   0.3148 0.1091 -0.0694 0.2972 0.5304 1.1834
2022        52560    52560      0   0.

### **3.4 Gráficos estatísticos**

In [8]:
import glob
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
  from google.colab import drive

  if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
except Exception:
  pass

# 1. Caminhos e parâmetros
pasta_base = '/content/drive/MyDrive/PUC/EQM2118'
pasta_consolidados = os.path.join(pasta_base, 'CONSOLIDADOS')
pasta_reconst = os.path.join(pasta_consolidados, 'SERIE_LAGOA_RECONST')
pasta_eda = os.path.join(pasta_reconst, 'ANALISE_EXPLORATORIA')
os.makedirs(pasta_eda, exist_ok=True)

LIMIARES = [0.80, 1.00, 1.20, 1.50]  # em metros (ajuste se desejar)

arquivos_reconst = sorted(
    glob.glob(os.path.join(pasta_reconst, 'serie_lagoa_reconstituida_*.csv'))
)
if not arquivos_reconst:
  raise FileNotFoundError(
      f'Nenhum arquivo reconstruído encontrado em {pasta_reconst}'
  )
caminho_reconst = arquivos_reconst[-1]

# 2. Carregar série reconstruída na grade regular de 10 min sem interpolar
df_rec = pd.read_csv(caminho_reconst)
col_dt = 'Date_time' if 'Date_time' in df_rec.columns else df_rec.columns[0]
col_na = 'NA_lagoa' if 'NA_lagoa' in df_rec.columns else df_rec.columns[1]

df_rec['dt'] = pd.to_datetime(df_rec[col_dt], errors='coerce')
df_rec = (
    df_rec.dropna(subset=['dt'])
    .sort_values('dt')
    .drop_duplicates(subset=['dt'], keep='last')
    .set_index('dt')
)

grade_10min = pd.date_range(
    start=df_rec.index.min(), end=df_rec.index.max(), freq='10min'
)
s_rec = pd.to_numeric(df_rec[col_na], errors='coerce').reindex(grade_10min)
s_val = s_rec.dropna()

# 3. Estatísticas anuais: média, mínimo e máximo
df_anual = (
    pd.DataFrame({'NA_lagoa': s_rec, 'ano': s_rec.index.year})
    .groupby('ano')['NA_lagoa']
    .agg(
        validos='count',
        min_m='min',
        media_m='mean',
        max_m='max',
    )
    .reset_index()
)

print('=== ESTATÍSTICAS ANUAIS (MÍNIMO, MÉDIA, MÁXIMO) ===')
print(df_anual.round(4).to_string(index=False))

# 4. Frequência acima de limiares (série reconstruída)
linhas_lim = []
for lim in LIMIARES:
  qtd = int((s_val > lim).sum())
  pct = float(qtd / len(s_val) * 100) if len(s_val) > 0 else 0.0
  horas_equiv = (
      qtd * 10 / 60.0
  )  # cada registro equivale a 10 minutos (sem interpolação)
  linhas_lim.append({
      'limiar_m': lim,
      'n_registros_10min': qtd,
      'pct_validos': round(pct, 4),
      'horas_equiv': round(horas_equiv, 2),
  })

df_lim = pd.DataFrame(linhas_lim)
df_lim.to_csv(
    os.path.join(pasta_eda, 'frequencia_limiares_reconstruida.csv'), index=False
)

print('\n=== FREQUÊNCIA ACIMA DE LIMIARES (RECONSTRUÍDA) ===')
print(df_lim.to_string(index=False))

# 5. Gráfico 1: Estatísticas anuais (Média, Mínimo e Máximo) — legenda na lateral direita
plt.rcParams['figure.dpi'] = 140
anos_str = df_anual['ano'].astype(str)
x = np.arange(len(anos_str))

fig, ax = plt.subplots(figsize=(12.5, 5.0))

# Faixa vertical entre mínimo e máximo anual
ax.vlines(
    x,
    df_anual['min_m'],
    df_anual['max_m'],
    color='#6baed6',
    linewidth=3.0,
    alpha=0.85,
    label='Amplitude anual (Mín. a Máx.)',
)
ax.scatter(
    x,
    df_anual['max_m'],
    color='#cb181d',
    s=55,
    zorder=4,
    label='Máximo anual',
)
ax.plot(
    x,
    df_anual['media_m'],
    color='#08519c',
    marker='o',
    linewidth=2.0,
    markersize=6,
    zorder=5,
    label='Média anual',
)
ax.scatter(
    x,
    df_anual['min_m'],
    color='#238b45',
    s=55,
    zorder=4,
    label='Mínimo anual',
)

# Margem vertical para os rótulos não encostarem nas bordas
ymax_limite = max(2.15, float(df_anual['max_m'].max()) + 0.22)
ymin_limite = min(-0.35, float(df_anual['min_m'].min()) - 0.12)
ax.set_ylim(ymin_limite, ymax_limite)

# Rótulos discretos de máximo e média
for i, row in df_anual.iterrows():
  ax.annotate(
      f"{row['max_m']:.2f}",
      (x[i], row['max_m']),
      textcoords='offset points',
      xytext=(0, 7),
      ha='center',
      fontsize=8,
      color='#99000d',
  )
  ax.annotate(
      f"{row['media_m']:.2f}",
      (x[i], row['media_m']),
      textcoords='offset points',
      xytext=(8, 0),
      ha='left',
      va='center',
      fontsize=8,
      color='#08306b',
  )

ax.axhline(0, color='gray', linestyle=':', linewidth=0.9)
ax.set_xticks(x)
ax.set_xticklabels(anos_str, rotation=35)
ax.set_xlabel('Ano')
ax.set_ylabel('Nível da lagoa (m)')
ax.set_title(
    'Estatísticas Anuais do NA_lagoa Reconstruído (Mínimo, Média e Máximo)'
)
ax.legend(
    loc='center left',
    bbox_to_anchor=(1.02, 0.5),
    ncol=1,
    frameon=True,
)
ax.grid(True, axis='y', alpha=0.25)
plt.tight_layout()

caminho_fig_anual = os.path.join(
    pasta_eda, 'estatisticas_anuais_media_min_max.png'
)
plt.savefig(caminho_fig_anual, bbox_inches='tight')
plt.close()

# 6. Gráfico 2: Frequência acima de limiares (Reconstruída)
labels_lim = [f'> {lim:.2f} m' for lim in df_lim['limiar_m']]
x_lim = np.arange(len(labels_lim))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.5))

barras1 = ax1.bar(
    x_lim,
    df_lim['n_registros_10min'],
    color='#2171b5',
    edgecolor='black',
    linewidth=0.6,
)
for b in barras1:
  h = b.get_height()
  ax1.annotate(
      f'{int(h)}',
      (b.get_x() + b.get_width() / 2, h),
      textcoords='offset points',
      xytext=(0, 4),
      ha='center',
      fontsize=8.5,
  )
ax1.set_xticks(x_lim)
ax1.set_xticklabels(labels_lim)
ax1.set_xlabel('Limiar de nível')
ax1.set_ylabel('Nº de registros (10 min)')
ax1.set_title('Frequência Absoluta Acima dos Limiares')
ax1.grid(True, axis='y', alpha=0.25)

barras2 = ax2.bar(
    x_lim,
    df_lim['pct_validos'],
    color='#e6550d',
    edgecolor='black',
    linewidth=0.6,
)
for b in barras2:
  h = b.get_height()
  ax2.annotate(
      f'{h:.3f}%',
      (b.get_x() + b.get_width() / 2, h),
      textcoords='offset points',
      xytext=(0, 4),
      ha='center',
      fontsize=8.5,
  )
ax2.set_xticks(x_lim)
ax2.set_xticklabels(labels_lim)
ax2.set_xlabel('Limiar de nível')
ax2.set_ylabel('Frequência relativa (% dos válidos)')
ax2.set_title('Frequência Relativa Acima dos Limiares')
ax2.grid(True, axis='y', alpha=0.25)

fig.suptitle(
    'Frequência Acima de Limiares — Série Reconstruída (2017–2026)',
    y=1.02,
    fontsize=11,
)
plt.tight_layout()

caminho_fig_lim = os.path.join(pasta_eda, 'frequencia_acima_limiares.png')
plt.savefig(caminho_fig_lim, bbox_inches='tight')
plt.close()

print(f'\nGráficos salvos em:\n  • {caminho_fig_anual}\n  • {caminho_fig_lim}')

=== ESTATÍSTICAS ANUAIS (MÍNIMO, MÉDIA, MÁXIMO) ===
 ano  validos   min_m  media_m  max_m
2017    15296 -0.0209   0.2929 1.2013
2018    52560 -0.0275   0.3194 1.6874
2019    52560 -0.0299   0.3167 1.9112
2020    52704 -0.0166   0.3108 1.4661
2021    52560 -0.0694   0.3148 1.1834
2022    52560 -0.0205   0.3196 1.5645
2023    52560 -0.0049   0.3151 1.3059
2024    52560 -0.1730   0.2972 1.2705
2025    52416 -0.2250   0.2866 1.7839
2026    37723 -0.1980   0.3308 1.6133

=== FREQUÊNCIA ACIMA DE LIMIARES (RECONSTRUÍDA) ===
 limiar_m  n_registros_10min  pct_validos  horas_equiv
      0.8               1677       0.3542       279.50
      1.0                612       0.1293       102.00
      1.2                329       0.0695        54.83
      1.5                102       0.0215        17.00

Gráficos salvos em:
  • /content/drive/MyDrive/PUC/EQM2118/CONSOLIDADOS/SERIE_LAGOA_RECONST/ANALISE_EXPLORATORIA/estatisticas_anuais_media_min_max.png
  • /content/drive/MyDrive/PUC/EQM2118/CONSOLIDADO